<a href="https://www.kaggle.com/code/g25ait2099/mlops-group-assignment-3" target="_blank">
  <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle">
</a>

# MLOps Group 14 - Emotion Classification (Sharvan)
**Tasks 3, 4 & 5: Model Selection, Kaggle Training, W&B Tracking, and HF Deployment**

This single notebook runs **both** of my experiment versions back to back and then pushes the
winning model to the Hugging Face Hub. The two versions differ by exactly one hyperparameter -
the per-device batch size (32 vs 16) - so the W&B comparison is clean. Unlike the first batch of
team notebooks, my `compute_metrics` logs **weighted F1 in addition to accuracy**, which the
Task 4 rubric asks for.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Environment Setup
We need `transformers` and `datasets` for the model and data, `scikit-learn` for the accuracy
and F1 metrics, and `wandb` to track every run on our shared dashboard.

In [ ]:
!pip install -q transformers datasets scikit-learn wandb

## 2. Secure Authentication
**Justification:** Hardcoding API keys is bad MLOps practice. I pull the W&B and Hugging Face
tokens from **Kaggle Secrets** (Add-ons -> Secrets). W&B is our central tracking dashboard and
the HF login lets me push the final model in Task 5. I set `WANDB_PROJECT` to the same value as
the rest of Group 14 so all of our runs appear together for the Task 8 comparison.

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
hf_token = user_secrets.get_secret("HF_TOKEN")

wandb.login()
os.environ["WANDB_PROJECT"] = "mlops-emotion-classification"

login(token=hf_token)
print(f"Logged in to Hugging Face as: {whoami()['name']}")

## 3. Data Loading & Model Selection (Task 3)
**Justification:**
* **Model:** `distilbert-base-uncased`. Per the HF model card it is a distilled BERT - ~40%
  smaller and ~60% faster while keeping ~97% of BERT's language understanding. At ~260MB it fine-
  tunes comfortably inside Kaggle's free T4 quota, which fits the assignment's compact-model rule.
* **Data:** `dair-ai/emotion` - 6 emotion classes. I keep the `id2label` map inline as the single
  source of truth for `num_labels`, and take a seeded 8k/2k subset so each run is short.

Because I train two models in this one notebook, I wrap model creation in `build_model()` so each
experiment starts from fresh, untrained weights instead of accidentally continuing the previous run.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Single source of truth for the label space (mirrors data/id2label.json in the repo)
id2label = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )


dataset = load_dataset("dair-ai/emotion")
print(dataset)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)


tokenized = dataset.map(tokenize, batched=True)
train_ds = tokenized["train"].shuffle(seed=42).select(range(8000))
eval_ds = tokenized["validation"].shuffle(seed=42).select(range(2000))
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

## 4. Shared metrics + a small training helper
`compute_metrics` returns **accuracy and weighted F1**; the Trainer already streams training loss
and `eval_loss`, so all four required metrics land on W&B. `run_experiment()` keeps the two runs
identical apart from the hyperparameters I pass in.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


def run_experiment(run_name, version, batch_size, epochs=3, lr=2e-5, weight_decay=0.01):
    run = wandb.init(
        project="mlops-emotion-classification",
        name=run_name,
        config={
            "model": model_name,
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "max_length": 128,
            "train_samples": len(train_ds),
            "version": version,
            "platform": "Kaggle",
        },
    )
    args = TrainingArguments(
        output_dir=f"./results_{version}",
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=25,
        report_to="wandb",
        run_name=run_name,
    )
    trainer = Trainer(
        model=build_model(),
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print(metrics)
    print(f"\n>> W&B run dashboard: {run.url}")
    wandb.finish()
    return trainer, metrics

## 5. Experiment 1 - batch size 32 (Task 4)
The larger-batch baseline. Standard DistilBERT fine-tuning settings: lr 2e-5, 3 epochs,
weight decay 0.01.

In [ ]:
trainer_v1, metrics_v1 = run_experiment("sharvan-exp1-bs32", version="v1", batch_size=32)

## 6. Experiment 2 - batch size 16 (Task 4)
**The only change** is batch size 32 -> 16. Smaller batches give more gradient updates per epoch,
which often nudges accuracy/F1 up on a small subset at the cost of a slightly noisier loss curve.

In [ ]:
trainer_v2, metrics_v2 = run_experiment("sharvan-exp2-bs16", version="v2", batch_size=16)

## 7. Push the winning model to Hugging Face (Task 5)
**Justification:** I pick whichever run has the higher validation F1 and push its weights +
tokenizer to our shared **public** group repo `zeeshan-hf/mlops-emotion-distilbert-group14`
(the same model the Docker / GitHub Actions inference pulls), then record the link in the W&B
run summary so the model is traceable from the dashboard.

In [ ]:
hf_repo = "zeeshan-hf/mlops-emotion-distilbert-group14"

best_trainer = trainer_v2 if metrics_v2["eval_f1"] >= metrics_v1["eval_f1"] else trainer_v1
best_metrics = metrics_v2 if best_trainer is trainer_v2 else metrics_v1
print(f"Best run F1 = {best_metrics['eval_f1']:.4f} - pushing to {hf_repo}")

best_trainer.model.push_to_hub(hf_repo)
tokenizer.push_to_hub(hf_repo)

hf_url = f"https://huggingface.co/{hf_repo}"
with wandb.init(project="mlops-emotion-classification", name="sharvan-best-model", job_type="deploy") as run:
    run.summary["huggingface_model"] = hf_url
    run.summary["best_accuracy"] = best_metrics["eval_accuracy"]
    run.summary["best_f1"] = best_metrics["eval_f1"]
    project_url = run.get_project_url()

print("\n================ LINKS ================")
print(f"Hugging Face model : {hf_url}")
print(f"W&B project board  : {project_url}")
print("=======================================")

## 8. Evaluate the best model on the held-out test split (Task 8)
The validation set drove model selection, so I report the final numbers on the **test** split,
which neither run ever saw. I reuse the best model already in memory (no re-download) and print
the figures behind the comparison table: accuracy, **weighted F1** (handles the class imbalance),
**macro F1** (treats every emotion equally), a per-class report and a confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

label_names = [id2label[i] for i in sorted(id2label)]
test_ds = tokenized["test"]

pred = best_trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print("==== Test-set results (best model) ====")
print(f"Samples      : {len(y_true)}")
print(f"Accuracy     : {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 (weighted): {f1_score(y_true, y_pred, average='weighted'):.4f}")
print(f"F1 (macro)   : {f1_score(y_true, y_pred, average='macro'):.4f}")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

print("Confusion matrix (rows = true, cols = predicted):")
print(label_names)
print(confusion_matrix(y_true, y_pred))